In [ ]:
# ============================================================================
# CAMINHO C — VISUALIZAÇÃO DOS PESOS DE ATENÇÃO ESPACIAL
# ============================================================================
# Objetivo: verificar se o SpatialAttentionModule aprendeu estrutura
# espacial no grid 7×7, mesmo que a atenção temporal seja uniforme.
#
# Perguntas:
#   1. Os spatial_weights mostram padrão claro (borda vs centro,
#      direção específica, vizinhança próxima vs distante)?
#   2. O padrão de atenção difere entre TP, FP, FN, TN?
#   3. O padrão de atenção difere entre conversão rápida (label=1)
#      e lenta (label=0)?
#   4. Há diferença de atenção por padrão espacial (Class8590)?
#
# Inputs:
#   - spatial_weights_*.npy   (shape: n_test × 5 × 49)
#   - temporal_weights_*.npy  (shape: n_test × 5)
#   - predictions_test.csv    (prob, pred, label, error_type, Class8590)
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

# ── Caminhos ────────────────────────────────────────────────────────────────
BASE_DIR   = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
FREEZE_DIR = BASE_DIR / "geofmv3_frozen"

SPATIAL_W  = FREEZE_DIR / "spatial_weights_20260423_071506.npy"
TEMPORAL_W = FREEZE_DIR / "temporal_weights_20260423_071506.npy"
PRED_CSV   = BASE_DIR / "multihead_spatial_frozen" / "predictions_test.csv"

# ── Carregar ────────────────────────────────────────────────────────────────
spat_w = np.load(SPATIAL_W)    # (n_test, 5, 49)
temp_w = np.load(TEMPORAL_W)   # (n_test, 5)
df     = pd.read_csv(PRED_CSV)

print(f"✅ Dados carregados!")
print(f"   spatial_weights:  {spat_w.shape}  → (amostras, anos, posições)")
print(f"   temporal_weights: {temp_w.shape}     → (amostras, anos)")
print(f"   predictions:      {df.shape}")
print()

# Verificar alinhamento
assert len(spat_w) == len(df), \
    f"Tamanhos não batem: {len(spat_w)} vs {len(df)}"

print(f"Distribuição de error_type:")
print(df['error_type'].value_counts())
print()
print(f"Distribuição de Class8590:")
print(df['Class8590'].value_counts())

In [ ]:
# ============================================================================
# ANÁLISE 1 — PADRÃO GERAL DE ATENÇÃO
# ============================================================================
# Pergunta: em média, o modelo olha para o centro ou para as bordas?
# Se há padrão, o módulo está aprendendo algo sobre vizinhança.
# Se for uniforme (todos ~1/49 ≈ 0.0204), não aprendeu estrutura espacial.

PATCH_N = 7
uniform = 1.0 / (PATCH_N * PATCH_N)  # 0.0204

# Média sobre amostras e anos → mapa 7×7 médio
mean_attn_map = spat_w.mean(axis=0).mean(axis=0).reshape(PATCH_N, PATCH_N)

# Por ano
mean_per_year = [spat_w[:, t, :].mean(axis=0).reshape(PATCH_N, PATCH_N)
                 for t in range(5)]

# Estatísticas de variação
flat = spat_w.mean(axis=0).mean(axis=0)  # média por posição
print(f"Atenção espacial média por posição:")
print(f"  Min:    {flat.min():.6f}  (uniforme seria {uniform:.6f})")
print(f"  Max:    {flat.max():.6f}")
print(f"  Std:    {flat.std():.6f}")
print(f"  Ratio max/uniforme: {flat.max()/uniform:.2f}x")
print()
if flat.std() < 0.001:
    print("⚠️  Atenção UNIFORME — o módulo não aprendeu estrutura espacial")
elif flat.std() < 0.003:
    print("⚠️  Atenção levemente não-uniforme — padrão fraco")
else:
    print("✅ Atenção NÃO-UNIFORME — o módulo aprendeu estrutura espacial")

# Gráfico
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

# Mapa médio geral
im = axes[0, 0].imshow(mean_attn_map, cmap='hot', vmin=0)
axes[0, 0].set_title('Atenção média (todos anos)', fontweight='bold')
axes[0, 0].set_xlabel('Col (Leste →)'); axes[0, 0].set_ylabel('Row (Norte ↑)')
# Marcar pixel central
axes[0, 0].plot(3, 3, 'c+', markersize=15, markeredgewidth=2,
                label='Pixel alvo')
axes[0, 0].legend()
plt.colorbar(im, ax=axes[0, 0])

# Por ano
year_labels = ['T-5', 'T-4', 'T-3', 'T-2', 'T-1']
ax_positions = [(0,1),(0,2),(1,0),(1,1),(1,2)]
for i, (t, label) in enumerate(zip(range(5), year_labels)):
    ax = axes[ax_positions[i]]
    im2 = ax.imshow(mean_per_year[t], cmap='hot', vmin=0)
    ax.set_title(f'Atenção ano {label}')
    ax.plot(3, 3, 'c+', markersize=12, markeredgewidth=2)
    plt.colorbar(im2, ax=ax)

plt.suptitle('Mapas de Atenção Espacial — GeoFM v3',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(FREEZE_DIR / 'attention_maps_overall.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Salvo: attention_maps_overall.png")

In [ ]:
# ============================================================================
# ANÁLISE 2 — ATENÇÃO POR TIPO DE ERRO
# ============================================================================
# Pergunta: o modelo olha para lugares diferentes quando erra vs acerta?
# FN vs TP é a comparação mais relevante:
#   - TP: conversão rápida, modelo acerta → o que o modelo está vendo?
#   - FN: conversão rápida, modelo erra  → o que o modelo está perdendo?

fig, axes = plt.subplots(2, 4, figsize=(16, 9))

error_types = ['TP', 'TN', 'FP', 'FN']
colors_title = {'TP': '#27ae60', 'TN': '#2980b9',
                'FP': '#e67e22', 'FN': '#c0392b'}

for col, etype in enumerate(error_types):
    idx = df[df['error_type'] == etype].index
    n   = len(idx)

    if n == 0:
        continue

    # Média da atenção para este tipo de erro
    attn_mean = spat_w[idx].mean(axis=0).mean(axis=0).reshape(PATCH_N, PATCH_N)

    # Linha 0: mapa de atenção
    im = axes[0, col].imshow(attn_mean, cmap='hot', vmin=0)
    axes[0, col].set_title(f'{etype} (n={n})',
                            color=colors_title[etype], fontweight='bold')
    axes[0, col].plot(3, 3, 'c+', markersize=12, markeredgewidth=2)
    plt.colorbar(im, ax=axes[0, col])

    # Linha 1: diferença em relação ao mapa médio geral
    diff = attn_mean - mean_attn_map
    vmax = max(abs(diff.min()), abs(diff.max()))
    im2  = axes[1, col].imshow(diff, cmap='RdBu_r',
                                vmin=-vmax, vmax=vmax)
    axes[1, col].set_title(f'{etype} − média geral',
                            color=colors_title[etype])
    axes[1, col].plot(3, 3, 'w+', markersize=12, markeredgewidth=2)
    plt.colorbar(im2, ax=axes[1, col])

axes[0, 0].set_ylabel('Atenção média', fontsize=11)
axes[1, 0].set_ylabel('Diferença vs média geral', fontsize=11)

plt.suptitle('Atenção Espacial por Tipo de Erro\n'
             '(azul=mais atenção que média, vermelho=menos atenção)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(FREEZE_DIR / 'attention_maps_error_type.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Salvo: attention_maps_error_type.png")

# Resumo numérico: atenção no pixel central vs bordas
print(f"\nAtenção no pixel central (posição [3,3]) por tipo de erro:")
print(f"{'Tipo':<6} {'n':>5} {'Centro':>8} {'Borda':>8} {'Ratio':>7}")
print("-" * 38)
for etype in error_types:
    idx = df[df['error_type'] == etype].index
    if len(idx) == 0: continue
    attn = spat_w[idx].mean(axis=0).mean(axis=0).reshape(PATCH_N, PATCH_N)
    centro = attn[3, 3]
    borda  = (attn[0,:].mean() + attn[-1,:].mean() +
              attn[:,0].mean() + attn[:,-1].mean()) / 4
    print(f"{etype:<6} {len(idx):>5} {centro:>8.5f} {borda:>8.5f} "
          f"{centro/borda:>7.2f}x")
print(f"  (ratio >1 = modelo olha mais para o centro; <1 = mais para as bordas)")

In [ ]:
# ============================================================================
# ANÁLISE 3 — ATENÇÃO POR LABEL E POR CLASS8590
# ============================================================================

fig, axes = plt.subplots(2, 5, figsize=(18, 8))

# Linha 0: por label (conversão rápida vs lenta)
for col, (lbl, title) in enumerate([(0, 'Lenta (label=0)'),
                                     (1, 'Rápida (label=1)')]):
    idx  = df[df['label'] == lbl].index
    attn = spat_w[idx].mean(axis=0).mean(axis=0).reshape(PATCH_N, PATCH_N)
    im   = axes[0, col].imshow(attn, cmap='hot', vmin=0)
    axes[0, col].set_title(f'{title}\n(n={len(idx)})', fontweight='bold')
    axes[0, col].plot(3, 3, 'c+', markersize=12, markeredgewidth=2)
    plt.colorbar(im, ax=axes[0, col])

# Diferença rápida − lenta
idx0 = df[df['label'] == 0].index
idx1 = df[df['label'] == 1].index
attn0 = spat_w[idx0].mean(axis=0).mean(axis=0).reshape(PATCH_N, PATCH_N)
attn1 = spat_w[idx1].mean(axis=0).mean(axis=0).reshape(PATCH_N, PATCH_N)
diff_label = attn1 - attn0
vmax = max(abs(diff_label.min()), abs(diff_label.max()))
im_d = axes[0, 2].imshow(diff_label, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0, 2].set_title('Rápida − Lenta\n(azul=mais atenção em rápida)',
                      fontweight='bold')
axes[0, 2].plot(3, 3, 'w+', markersize=12, markeredgewidth=2)
plt.colorbar(im_d, ax=axes[0, 2])

# Ocultar subplots não usados na linha 0
axes[0, 3].axis('off')
axes[0, 4].axis('off')

# Linha 1: por padrão Class8590 (top 5 por frequência)
top_patterns = df['Class8590'].value_counts().head(5).index.tolist()
for col, pat in enumerate(top_patterns):
    idx  = df[df['Class8590'] == pat].index
    attn = spat_w[idx].mean(axis=0).mean(axis=0).reshape(PATCH_N, PATCH_N)
    im   = axes[1, col].imshow(attn, cmap='hot', vmin=0)
    axes[1, col].set_title(f'{pat}\n(n={len(idx)})')
    axes[1, col].plot(3, 3, 'c+', markersize=10, markeredgewidth=2)
    plt.colorbar(im, ax=axes[1, col])

plt.suptitle('Atenção Espacial: por Label e por Padrão Espacial (Class8590)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(FREEZE_DIR / 'attention_maps_label_pattern.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Salvo: attention_maps_label_pattern.png")

In [ ]:
# ============================================================================
# ANÁLISE 4 — VARIAÇÃO INDIVIDUAL DOS PESOS
# ============================================================================
# Os mapas médios podem esconder variação individual importante.
# Se cada amostra tem um mapa muito diferente → o módulo está fazendo
# routing sample-specific (bom sinal).
# Se todos os mapas são quase iguais → o módulo aprendeu um filtro fixo.

# Variância por posição (média sobre anos)
attn_per_sample = spat_w.mean(axis=1)  # (n_test, 49)
var_per_pos = attn_per_sample.var(axis=0).reshape(PATCH_N, PATCH_N)
std_per_pos = attn_per_sample.std(axis=0).reshape(PATCH_N, PATCH_N)

print(f"Variação dos pesos de atenção entre amostras:")
print(f"  Std média por posição: {std_per_pos.mean():.6f}")
print(f"  Std máx por posição:   {std_per_pos.max():.6f}")
print(f"  Std do pixel central:  {std_per_pos[3,3]:.6f}")
print()
if std_per_pos.mean() < 0.005:
    print("⚠️  Baixa variação — módulo aprendeu filtro FIXO")
    print("   Mesma atenção para todas as amostras")
else:
    print("✅ Alta variação — módulo faz routing SAMPLE-SPECIFIC")
    print("   Atenção diferente por amostra (desejável)")

# Gráfico
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

im1 = axes[0].imshow(mean_attn_map, cmap='hot')
axes[0].set_title('Média', fontweight='bold')
axes[0].plot(3, 3, 'c+', markersize=12, markeredgewidth=2)
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(std_per_pos, cmap='viridis')
axes[1].set_title('Desvio padrão entre amostras', fontweight='bold')
axes[1].plot(3, 3, 'w+', markersize=12, markeredgewidth=2)
plt.colorbar(im2, ax=axes[1])

# Distribuição de atenção no pixel central
center_attn = attn_per_sample[:, 3*7+3]  # posição [3,3] no flat
axes[2].hist(center_attn, bins=30, color='steelblue', edgecolor='white')
axes[2].axvline(x=uniform, color='red', linestyle='--',
                label=f'Uniforme ({uniform:.4f})')
axes[2].axvline(x=center_attn.mean(), color='green', linestyle='-',
                label=f'Média ({center_attn.mean():.4f})')
axes[2].set_xlabel('Peso de atenção no pixel central')
axes[2].set_ylabel('N amostras')
axes[2].set_title('Distribuição: atenção no pixel central')
axes[2].legend()

plt.suptitle('Variação Individual dos Pesos de Atenção Espacial',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(FREEZE_DIR / 'attention_variation.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Salvo: attention_variation.png")

In [ ]:
# ============================================================================
# RESUMO DIAGNÓSTICO — Caminho C
# ============================================================================

print("=" * 65)
print("RESUMO DIAGNÓSTICO — ATENÇÃO ESPACIAL GeoFM v3")
print("=" * 65)

# 1. Uniformidade geral
flat_all = spat_w.mean(axis=0).mean(axis=0)
is_uniform_spatial = flat_all.std() < 0.002
print(f"\n1. Padrão geral de atenção:")
print(f"   Std por posição: {flat_all.std():.6f} "
      f"({'UNIFORME ⚠️' if is_uniform_spatial else 'NÃO-UNIFORME ✅'})")
print(f"   Posição mais atendida: {flat_all.argmax()} "
      f"→ [{flat_all.argmax()//7}, {flat_all.argmax()%7}] "
      f"(centro seria [3,3]=24)")

# 2. Variação entre amostras
print(f"\n2. Variação entre amostras:")
print(f"   Std médio: {std_per_pos.mean():.6f} "
      f"({'fixo ⚠️' if std_per_pos.mean() < 0.005 else 'sample-specific ✅'})")

# 3. Diferença TP vs FN
idx_tp = df[df['error_type']=='TP'].index
idx_fn = df[df['error_type']=='FN'].index
attn_tp = spat_w[idx_tp].mean(axis=0).mean(axis=0)
attn_fn = spat_w[idx_fn].mean(axis=0).mean(axis=0)
diff_tp_fn = np.abs(attn_tp - attn_fn).mean()
print(f"\n3. Diferença TP vs FN:")
print(f"   Diferença média |TP−FN|: {diff_tp_fn:.6f} "
      f"({'distinto ✅' if diff_tp_fn > 0.001 else 'similar ⚠️'})")

# 4. Atenção temporal
temp_std = temp_w.std(axis=0).mean()
print(f"\n4. Atenção temporal:")
print(f"   Média por ano: {temp_w.mean(axis=0).round(4).tolist()}")
print(f"   Std entre amostras (média): {temp_std:.6f} "
      f"({'uniforme ⚠️' if temp_std < 0.01 else 'discriminativa ✅'})")

# 5. Diagnóstico final
print(f"\n{'='*65}")
print(f"DIAGNÓSTICO FINAL:")
print(f"{'='*65}")

if is_uniform_spatial and std_per_pos.mean() < 0.005:
    print("""
  MÓDULO DE ATENÇÃO NÃO APRENDEU ESTRUTURA ESPACIAL.

  Tanto a atenção espacial quanto a temporal são uniformes.
  A melhoria de performance (+1.8pp) provavelmente vem do
  encoder espacial por ano (processa cada frame 7×7 separadamente)
  e não dos mecanismos de atenção em si.

  → Recomendação: Caminho B — revisar arquitetura de atenção.
""")
elif is_uniform_spatial and not std_per_pos.mean() < 0.005:
    print("""
  ATENÇÃO MÉDIA UNIFORME MAS COM VARIAÇÃO INDIVIDUAL.

  O módulo faz routing sample-specific, mas em média cancela.
  Pode ser que diferentes amostras usem diferentes regiões do patch.

  → Investigar se a variação individual correlaciona com label ou padrão.
""")
else:
    print("""
  MÓDULO APRENDEU ESTRUTURA ESPACIAL.

  Atenção não-uniforme indica que o módulo discrimina posições.
  Verificar se o padrão é interpretável (borda vs centro,
  direção de expansão, etc.)

  → Continuar com análise de interpretabilidade.
""")

print(f"\nGráficos salvos em: {FREEZE_DIR}")
print(f"  attention_maps_overall.png")
print(f"  attention_maps_error_type.png")
print(f"  attention_maps_label_pattern.png")
print(f"  attention_variation.png")